In [1]:
import torch
import torch.nn as nn
import gymnasium as gym
import numpy as np
import mujoco

from cobot_BC_env import CobotBcEnv

from torch.utils.tensorboard import SummaryWriter
import time
from lerobot.datasets import LeRobotDataset
from lerobot.processor.normalize_processor import NormalizerProcessorStep
from lerobot.processor.normalize_processor import UnnormalizerProcessorStep
from torch.utils.data import DataLoader, Subset

/home/aaron-dsouza/programming/miniconda3/envs/mujoco/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class MlpNetwork(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dims):
        super().__init__()
        self.network= nn.Sequential(
            nn.Linear(state_dim, hidden_dims[0]),
            nn.ReLU(),
            nn.Linear(hidden_dims[0], hidden_dims[1]),
            nn.ReLU(),
            nn.Linear(hidden_dims[1], action_dim)
        )

    def forward(self, state):
        action = self.network(state)
        return action

In [3]:
class Dataset:

    def __init__(self, root = "./datasets/pick_place_ik_test",
                 repo_id = 'aaron/pick_place_ik_test',
                 val_split=0.1):
        
        full_dataset = LeRobotDataset(
            repo_id=repo_id,
            root=root,
            download_videos=False,
        )

        episode_indices = np.array(full_dataset.hf_dataset["episode_index"])
        episodes = np.unique(episode_indices)

        rng = np.random.default_rng(42)
        rng.shuffle(episodes)

        n_val = max(1, int(len(episodes) * val_split))

        val_episodes = episodes[:n_val]
        train_episodes = episodes[n_val:]

        train_indices = np.where(np.isin(episode_indices, train_episodes))[0]
        val_indices = np.where(np.isin(episode_indices, val_episodes))[0]

        self.train_dataset = Subset(full_dataset, train_indices)
        self.val_dataset = Subset(full_dataset, val_indices)

        # 5. Create native PyTorch Dataloaders
        self.train_loader = DataLoader(
            self.train_dataset, 
            batch_size=32, 
            shuffle=True
        )
        
        self.val_loader = torch.utils.data.DataLoader(
            self.val_dataset, 
            batch_size=32, 
            shuffle=False  
        )

        self.normalizer =  NormalizerProcessorStep.from_lerobot_dataset(
            dataset=full_dataset,
            features={
                "observation.state": full_dataset.features["observation.state"],
                "action": full_dataset.features["action"],
            },
            norm_map={
                "ACTION": "MEAN_STD",          
                "STATE": "MEAN_STD" 
            },
            device = "cpu"
        )

        self.unnormalizer =  UnnormalizerProcessorStep.from_lerobot_dataset(
            dataset=full_dataset,
            features={
                "observation.state": full_dataset.features["observation.state"],
                "action": full_dataset.features["action"],
            },
            norm_map={
                "ACTION": "MEAN_STD",          
                "STATE": "MEAN_STD" 
            },
            device = "cpu"
        )

In [4]:
class BehaviorCloning:
    def __init__(self, CobotCubeEnv, dataset):
        self.env = CobotCubeEnv

        state_dim = 7+7+3+3+3+3+1 #27
        action_dim = 7
        hidden_dims = (256, 256)

        self.policy = MlpNetwork(state_dim, action_dim, hidden_dims)

        self.writer = SummaryWriter(f"run/{int(time.time())}")

        self.policy_optimizer = torch.optim.Adam(self.policy.parameters(), lr=1e-3)
        self.criterion = nn.MSELoss()
        self.dataset = dataset

    def get_val_loss(self):
        self.policy.eval()
        running_loss = 0.0

        with torch.no_grad():
            for batch in self.dataset.val_loader:
                batch = self.dataset.normalizer(batch)
                obs = batch['observation.state']
                action = batch['action']

                pred_action = self.policy(obs)
                action_loss = self.criterion(pred_action, action)

                running_loss += action_loss.item()

        avg_val_loss = running_loss / len(dataset.val_loader)
        return avg_val_loss

    def train(self, epochs):
        self.policy.to('cpu')

        for epoch in range(epochs):
            self.policy.train()
            running_loss = 0.0
            for batch in dataset.train_loader:
                batch = dataset.normalizer(batch)

                obs = batch['observation.state']
                action = batch['action']

                pred_action = self.policy(obs)

                action_loss = self.criterion(pred_action, action)
                self.policy_optimizer.zero_grad()
                action_loss.backward()
                self.policy_optimizer.step()
                running_loss += action_loss.item()

            train_loss = running_loss / len(dataset.train_loader)
            val_loss = self.get_val_loss()

            self.writer.add_scalar("Loss/Train", train_loss, epoch)
            self.writer.add_scalar("Loss/Val", val_loss, epoch)
            print(f"Epoch {epoch}: Train Loss = {train_loss:.4f} | Val Loss = {val_loss:.4f}")

        self.writer.close()

    @torch.no_grad()
    def rollout(self):
        self.env.reset()
        self.policy.eval()
        for i in range(500):
            raw_state = self.env.get_observation()
            state_dict = {"observation.state": torch.from_numpy(raw_state).float().to('cpu')}
            state = self.dataset.normalizer(state_dict)["observation.state"]

            normalized_action = self.policy(state)
            
            action_dict = {"action": normalized_action}
            action = self.dataset.unnormalizer(action_dict)["action"].cpu().numpy()

            self.env.apply_action(action)
            self.env.render()

In [5]:
if __name__ == "__main__":
    dataset = Dataset(root = "./datasets/pick_place_1",
                      repo_id = 'aaron/pick_place_1')
    model = BehaviorCloning(CobotBcEnv(render_mode="human"), dataset)
    model.train(50)

Epoch 0: Train Loss = 0.1591 | Val Loss = 0.0836
Epoch 1: Train Loss = 0.0817 | Val Loss = 0.0725
Epoch 2: Train Loss = 0.0723 | Val Loss = 0.0632
Epoch 3: Train Loss = 0.0681 | Val Loss = 0.0629
Epoch 4: Train Loss = 0.0655 | Val Loss = 0.0628
Epoch 5: Train Loss = 0.0637 | Val Loss = 0.0654
Epoch 6: Train Loss = 0.0618 | Val Loss = 0.0577
Epoch 7: Train Loss = 0.0610 | Val Loss = 0.0569
Epoch 8: Train Loss = 0.0594 | Val Loss = 0.0620
Epoch 9: Train Loss = 0.0590 | Val Loss = 0.0554
Epoch 10: Train Loss = 0.0581 | Val Loss = 0.0539
Epoch 11: Train Loss = 0.0574 | Val Loss = 0.0543
Epoch 12: Train Loss = 0.0568 | Val Loss = 0.0566
Epoch 13: Train Loss = 0.0561 | Val Loss = 0.0541
Epoch 14: Train Loss = 0.0557 | Val Loss = 0.0525
Epoch 15: Train Loss = 0.0550 | Val Loss = 0.0536
Epoch 16: Train Loss = 0.0546 | Val Loss = 0.0538
Epoch 17: Train Loss = 0.0541 | Val Loss = 0.0516
Epoch 18: Train Loss = 0.0539 | Val Loss = 0.0523
Epoch 19: Train Loss = 0.0535 | Val Loss = 0.0526
Epoch 20: 

In [6]:
# model.train(10)

In [14]:
model.rollout()

In [8]:
torch.save({
    "actor": model.policy.state_dict(),
    "actor_optimizer": model.policy_optimizer.state_dict(),
}, "checkpoints/bc_pick_place_v1.pt")